In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/q16_rewrite/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Predefine the sizes we care about
sizes = [49, 14, 23, 45, 19, 3, 36, 9]

# 1) Build the set of supplier keys to exclude – use a GPU‐friendly regex and .unique()
supplier_keys = (
    supplier.S_SUPPKEY[
        supplier.S_COMMENT.str.contains(r"Customer.*Complaints")
    ]
    .unique()
)

# 2) Pre‐filter partsupp to drop any bad suppliers before the join
partsupp_filtered = (
    partsupp
    [~partsupp.PS_SUPPKEY.isin(supplier_keys)]
    [["PS_PARTKEY", "PS_SUPPKEY"]]
)

# 3) Filter part with vectorized predicates and .str.startswith() instead of a regex
part_filtered = (
    part
    [(part.P_BRAND != "Brand#45")
     & (~part.P_TYPE.str.startswith("MEDIUM POLISHED"))
     & part.P_SIZE.isin(sizes)]
    [["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]]
)

# 4) Join the two small, pre‐filtered tables
total = (
    part_filtered
    .merge(partsupp_filtered,
           left_on="P_PARTKEY", right_on="PS_PARTKEY")
    [["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
)

# 5) GPU‐accelerated groupby + nunique, then sort
total = (
    total
    .groupby(["P_BRAND", "P_TYPE", "P_SIZE"]).PS_SUPPKEY
    .nunique()
    .reset_index(name="SUPPLIER_CNT")
    .sort_values(
        by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
        ascending=[False, True, True, True]
    )
)


CPU times: user 91.6 ms, sys: 11.9 ms, total: 104 ms
Wall time: 104 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q16/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_3.pickle